In [ ]:
"""A notebook illustrates and tests basic cleaning functionality"""

In [ ]:
from src.pipelines.ipums_cleaning_example import run_cps_cleaning_example

In [ ]:
df = run_cps_cleaning_example(year=2019)

In [ ]:
df.head(10)

In [ ]:
from src.cleaning.context import CleaningContext

In [ ]:
from src.config.settings import settings

config_dir = settings.cleaning_config_root / "cps"
context = CleaningContext.from_config(
    config_dir=config_dir,
    source="ipums_cps_asec",
)
context.topcode["wage"], sorted(context.deflators)

## Wage topcode adjustment and deflators

The production pipeline (`config/cleaning/cps/pipeline.yaml`) now applies two more steps
after the universe filters and the `AGE` cap:

- **`wage_topcode`** (`TopcodeAdjuster`): multiplies `INCWAGE` by 1.5 for rows whose value
  is topcoded, using the year-banded thresholds in `config/cleaning/cps/topcode/wage.yaml`.
- **`deflator_merge`** (`DeflatorMergeStep`): adds `CPI_DEFLATOR`/`GDP_DEFLATOR`, looked up
  by income year (`YEAR - 1`) against `config/cleaning/cps/deflators/cpi.yaml` and
  `gdp_pce.yaml` - both transcribed exactly from `src.harmonization.cps_tables`
  (`CPI_DEFLATOR`/`GDP_PCE_DEFLATOR`), so an edit to either table now changes
  `context.compute_hash()` too.

Both tables only cover **1962-2009** (topcodes) and **income year 1961-2008** (deflators) -
years outside that range are explicitly skipped (`uncovered_years: skip`) and reported as a
`StepReport` warning rather than silently left null. `run_cps_cleaning_example`'s default
`year=2019` above falls entirely outside both ranges, so `CPI_DEFLATOR`/`GDP_DEFLATOR` are
null and `wage_topcode` is a no-op for that run - the cells below use `year=1996`, which both
tables do cover, to show the steps actually doing something.

In [ ]:
df_1996 = run_cps_cleaning_example(year=1996)
df_1996.select("AGE", "INCWAGE", "CPI_DEFLATOR", "GDP_DEFLATOR").head(10)

In [ ]:
import polars as pl

# CPI_DEFLATOR converts nominal INCWAGE into 2008 dollars (see
# src.harmonization.cps_tables.CPI_DEFLATOR's docstring).
df_1996.select(
    "AGE",
    "INCWAGE",
    (pl.col("INCWAGE") * pl.col("CPI_DEFLATOR")).alias("INCWAGE_2008_DOLLARS"),
).head(10)

### A data-quality caveat surfaced by wiring `wage_topcode` in

Running `wage_topcode` against real 1996 data shows **~37% of rows hit the topcode
multiplier** (`gte_match` in the `StepReport.branches_taken` logged above) - implausibly
high for genuine income-censoring correction, which normally affects a small percentage of
a sample. The underlying table explains why:

```python
from src.harmonization.cps_tables import MAXWG_TABLE
{y: MAXWG_TABLE[y] for y in range(1993, 2004)}
# {1993: 99999, 1994: 99999, 1995: 99999,
#  1996: 25000, 1997: 25000, ..., 2002: 25000,
#  2003: 35000, ...}
```

The `>=` threshold **drops 4x (99999 -> 25000) between 1995 and 1996**, stays there through
2002, then only partially recovers to 35000 - a pattern that doesn't match how nominal wage
topcode thresholds should move over time. This looks like a transcription issue in
`MAXWG_TABLE` (and therefore in `config/cleaning/cps/topcode/wage.yaml`, which folds the same
values in) rather than a real feature of the source `aa_clean` methodology, but the original
`.do` files aren't in this repo to verify against. **Treat `wage_topcode` output for
1996-2002 as unverified until someone checks this table against `aa_clean/clean7909km.do`
directly.**

In [ ]:
from src.harmonization.cps_tables import MAXWG_TABLE

threshold_1996 = MAXWG_TABLE[1996]
hit_rate = (df_1996["INCWAGE"] >= threshold_1996 * 1.5).mean()
print(f"1996 wage topcode threshold: {threshold_1996}")
print(f"share of df_1996 rows at/above the post-multiplier value: {hit_rate:.1%}")